# 01 · Bag of Words

**Duración estimada:** 25 minutos

## Propósito

Hacer con una librería lo que en el `00` hicimos a mano, y verificar que dé lo
mismo.

Después vamos a mirar qué decisiones toma esa librería que nosotros no tomamos, y
qué pasa cuando el vocabulario se llena de palabras que no dicen nada.

## Qué consume

Nada. Seguimos con las mismas cuatro oraciones.

## 1. El mismo problema, la misma solución

**Bag of words** —bolsa de palabras— es el nombre de lo que hicimos en el `00`:
tratar un documento como una bolsa donde están todas sus palabras sueltas, sin
orden.

`scikit-learn` es una librería de aprendizaje automático, y trae una herramienta
que hace exactamente esos tres pasos.

In [ ]:
# scikit-learn es una librería grande. Importamos solo la parte que necesitamos:
# la herramienta que cuenta palabras y arma la matriz.
from sklearn.feature_extraction.text import CountVectorizer

# pandas, para mostrar las tablas.
import pandas as pd

documentos = [
    "El perro muerde al hombre",
    "El hombre muerde al perro",
    "El perro come carne",
    "El hombre come comida",
]

### Antes de usarla: qué es esta herramienta

`CountVectorizer` es una **clase**: un objeto que primero se configura y después
se usa. Se trabaja en dos momentos.

| momento | método | qué hace |
|---|---|---|
| aprender | `.fit()` | recorre los documentos y arma el vocabulario |
| aplicar | `.transform()` | cuenta las palabras del vocabulario en cada documento |

Y hay un atajo, `.fit_transform()`, que hace las dos cosas de una.

Esa separación importa: el vocabulario se aprende **una vez**, con un conjunto de
documentos, y después se aplica a otros. Si mañana llega un documento nuevo con
una palabra que no estaba, esa palabra **no va a tener columna**.

Hagámoslo en dos pasos para verlo.

In [ ]:
contador = CountVectorizer()

# Paso 1: aprender el vocabulario.
contador.fit(documentos)

vocabulario_sklearn = contador.get_feature_names_out()

print("palabras distintas:", len(vocabulario_sklearn))
print(list(vocabulario_sklearn))

**Es el mismo vocabulario que armamos a mano en el `00`**, y en el mismo
orden alfabético.

Ahora el paso 2.

In [ ]:
# Paso 2: contar.
matriz = contador.transform(documentos)

print("tipo de lo que devuelve:", type(matriz).__name__)
print("forma:", matriz.shape, "-> 4 documentos por 8 palabras")

### `sparse matrix`: por qué no es una tabla normal

Lo que devolvió no es una tabla de números: es una **matriz esparsa**.

En estos cuatro documentos la matriz tiene 32 casilleros y 15 están en cero. En un
corpus real la proporción es mucho peor: con 10.000 documentos y 50.000 palabras
distintas, más del 99% de los casilleros son cero, porque ningún documento usa
todo el vocabulario.

Una matriz esparsa **guarda solamente los casilleros que no son cero**. Ocupa
mucho menos memoria, y a cambio no se puede mirar directamente.

Para verla hay que convertirla a una tabla normal con `.toarray()`.

In [ ]:
tabla_de_numeros = matriz.toarray()

nombres_de_fila = ["Doc 1", "Doc 2", "Doc 3", "Doc 4"]
tabla = pd.DataFrame(tabla_de_numeros, columns=vocabulario_sklearn, index=nombres_de_fila)
tabla

Misma tabla que la del `00`, en tres líneas en vez de quince.

Eso es lo que hace una librería: no inventa nada nuevo, hace más rápido y con
menos errores algo que ya sabemos hacer. Por eso convenía hacerlo a mano primero.

## 2. Las decisiones que toma la librería sin avisar

Acá hay algo importante. `CountVectorizer` **no** hace exactamente lo que hicimos
nosotros. Toma decisiones propias, y conviene saber cuáles.

Probemos con un texto que tenga puntuación y palabras de una letra.

In [ ]:
prueba = ["El perro come carne. Y a mí me gusta el pan."]

contador_prueba = CountVectorizer()
contador_prueba.fit(prueba)

print("nuestro .split() habría dado:")
print(prueba[0].lower().split())
print()
print("CountVectorizer da:")
print(list(contador_prueba.get_feature_names_out()))

Tres diferencias, y ninguna está escrita en el nombre de la herramienta:

1. **Sacó la puntuación.** `carne.` quedó como `carne`.
2. **Descartó las palabras de una sola letra.** `y`, `a` y `mí`… fijate cuáles
   sobrevivieron y cuáles no.
3. **Pasó todo a minúsculas**, igual que nosotros.

Ese comportamiento se puede cambiar, y está documentado.

In [ ]:
help(CountVectorizer.__init__)

Es mucho texto, y esa es la situación normal: la documentación de una
librería está escrita para quien ya sabe qué está buscando.

Los tres parámetros que explican lo que vimos son `lowercase`,
`token_pattern` y `stop_words`. El que descarta las palabras de una letra es
`token_pattern`, cuyo valor por omisión pide **dos caracteres o más**.

Comprobémoslo cambiándolo.

In [ ]:
# El patrón por omisión es r"(?u)\b\w\w+\b": dos caracteres o más.
# Lo cambiamos a uno o más.
contador_con_una_letra = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
contador_con_una_letra.fit(prueba)

print(list(contador_con_una_letra.get_feature_names_out()))

Ahora aparecen `a` e `y`.

> **Esto no es un detalle técnico.** Si contás palabras de un corpus con la
> configuración por omisión, estás afirmando que las palabras de una letra no
> importan. Puede estar bien o puede estar mal —depende de qué estés analizando—,
> pero es una afirmación tuya, no de la librería.

## 3. Stopwords

Volvamos a nuestra tabla y miremos qué palabra es la más frecuente.

In [ ]:
suma_por_palabra = tabla.sum()

print(suma_por_palabra.sort_values(ascending=False))

`el` aparece en los cuatro documentos y es la palabra más frecuente. No nos
dice nada sobre de qué hablan los textos.

Las palabras así —artículos, preposiciones, conjunciones— se llaman **stopwords**
o palabras vacías. Se las suele sacar antes de contar.

`CountVectorizer` acepta una lista.

In [ ]:
palabras_vacias = ["el", "al", "la", "los", "las", "un", "una", "de", "y"]

contador_sin_vacias = CountVectorizer(stop_words=palabras_vacias)
matriz_sin_vacias = contador_sin_vacias.fit_transform(documentos)

tabla_sin_vacias = pd.DataFrame(
    matriz_sin_vacias.toarray(),
    columns=contador_sin_vacias.get_feature_names_out(),
    index=nombres_de_fila,
)
tabla_sin_vacias

El vocabulario bajó de ocho palabras a seis, y las que quedaron dicen algo.

> **¿De dónde salió esa lista?** La escribí yo, hace treinta segundos, a ojo.
>
> Esa es la situación habitual: las listas de stopwords las escribió alguien, con
> algún criterio, para algún corpus. No son una propiedad del idioma.
>
> En el práctico vamos a usar una lista de 260 palabras y vamos a discutir qué
> pasa cuando una palabra que parece vacía resulta ser justamente el dato.

## 4. Lo que se pierde

Volvamos a los documentos 1 y 2, que en el `00` tenían vectores idénticos.

¿Sacar las stopwords lo arregló?

In [ ]:
fila_1 = tabla_sin_vacias.loc["Doc 1"]
fila_2 = tabla_sin_vacias.loc["Doc 2"]

print("Doc 1:", documentos[0])
print("Doc 2:", documentos[1])
print()
print("¿los vectores son iguales?", list(fila_1) == list(fila_2))

**No lo arregló, y no lo puede arreglar.**

El problema no son las stopwords. El problema es que bag of words **no guarda el
orden**, y quién muerde a quién es una cuestión de orden.

Esto es una limitación del modelo, no un error de configuración. Hay dos maneras
de atacarla, y vemos las dos:

- **N-gramas**, en el cuaderno siguiente: contar pares de palabras en vez de
  palabras sueltas.
- **Análisis sintáctico**, en la unidad de spaCy: preguntarle al texto quién es el
  sujeto del verbo.

La segunda es la que realmente resuelve el problema. La primera lo esquiva.

> ### Perilla
>
> `CountVectorizer` tiene un parámetro `binary`. Si lo ponés en `True`, en vez de
> contar cuántas veces aparece cada palabra, anota solamente **si aparece o no**.
>
> Corré la celda y compará las dos tablas. Después pensá: ¿para qué pregunta
> conviene contar, y para qué pregunta alcanza con saber si aparece?

In [ ]:
contador_binario = CountVectorizer(binary=True)
matriz_binaria = contador_binario.fit_transform(documentos)

tabla_binaria = pd.DataFrame(
    matriz_binaria.toarray(),
    columns=contador_binario.get_feature_names_out(),
    index=nombres_de_fila,
)
tabla_binaria

## Lo que hiciste hoy

- Reprodujiste con `CountVectorizer` lo que en el `00` hiciste a mano.
- Entendiste la separación entre `fit` (aprender el vocabulario) y `transform`
  (aplicarlo).
- Viste qué es una matriz esparsa y por qué existe.
- Descubriste tres decisiones que la librería toma por omisión, y dónde se
  cambian.
- Sacaste stopwords, y viste que la lista la escribe alguien.
- Confirmaste que bag of words pierde el orden y que eso no tiene arreglo dentro
  del modelo.

## Lo que viene

En el `02`, el problema que queda: en la tabla, todas las palabras pesan igual.
`carne` —que aparece en un solo documento— vale lo mismo que `el`, que aparece en
todos. TF-IDF intenta arreglar eso.